In [1]:
# %%capture
%matplotlib inline

import os
import warnings
import logging
import time
from pathlib import Path
from typing import Optional, List, Dict

# --- Silence TensorFlow and add-ons warnings for a clean log ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=UserWarning, module='tensorflow_addons')
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# --- Project Imports ---
from forecast_pipeline.config import LOG_LEVEL, DEFAULT_EXP_PARAMS, MAX_WORKERS, DEFAULT_DATASET
from forecast_pipeline.jobs import generate_jobs, select_data_sources, create_filter_configurations
from forecast_pipeline.metrics import clean_and_structure_results
from forecast_pipeline.io_utils import generate_experiment_name, save_experiment_to_excel, configure_logging
from forecast_pipeline.runner import run_experiments_for_config
from common.config_wells import DATA_SOURCES

INFO: Carregadas 6 arquiteturas e 5 perfis de hiperparâmetros.


In [2]:
# --- Main Pipeline Function ---
def main_legacy(
    ensemble_models: int = 1,
    filter_methods: Optional[List] = None,
    selected_sources: Optional[List] = None,
) -> Dict:
    """
    End-to-end pipeline for the legacy run mode.
    Selects data sources, iterates configs, runs jobs, and collects results.
    """
    logging.info("Starting legacy pipeline...")
    sources = select_data_sources(DATA_SOURCES, selected_sources)
    if not sources:
        logging.warning("No data sources selected; exiting.")
        return {}

    results = {}
    for cfg in create_filter_configurations(filter_methods):
        # Legacy runner always uses in-memory results collation
        results.update(run_experiments_for_config(
            cfg, sources, ensemble_models, profile_path=None
        ))

    logging.info("Legacy pipeline finished.")
    return results

# --- Entry Point ---
def run_legacy_pipeline():
    configure_logging()
    ensemble_size = 1
    start_time = time.time()

    # Run pipeline
    results = main_legacy(
        ensemble_models=ensemble_size,
        filter_methods=None,
        selected_sources=DEFAULT_DATASET,
    )

    # --- Result Saving ---
    exp_name = generate_experiment_name(
        DEFAULT_DATASET,
        DEFAULT_EXP_PARAMS["architecture_name"],
        ensemble_size
    )
    output_path = save_experiment_to_excel(
        DEFAULT_EXP_PARAMS,
        results,
        exp_name,
        DEFAULT_DATASET,
        ensemble_size,
    )
    logging.info(f"Results saved to: {output_path}")

    elapsed_time = time.time() - start_time
    logging.info(f"Execution time: {elapsed_time:.2f} seconds")

# --- Run if script is called directly ---
if __name__ == "__main__":
    run_legacy_pipeline()


2025-07-23 12:51:07 [INFO   ] Starting legacy pipeline...
2025-07-23 12:51:07 [INFO   ] No filter methods provided; running with no adaptive filtering.
2025-07-23 12:51:07 [WARNING] No profile path provided. Falling back to legacy job generation.
2025-07-23 12:51:07 [INFO   ] Dispatching 5 jobs


Jobs:   0%|          | 0/5 [00:00<?, ?job/s]

2025-07-23 12:51:07 [INFO   ] → Beginning process_chunks: total=1, chunk=1
2025-07-23 12:51:07 [INFO   ]   → Batch 1/1 (size=1)
2025-07-23 12:51:07 [INFO   ] → Launching chunk of 1 models
2025-07-23 12:51:07 [INFO   ] Criando modelo para arquitetura: 'Seq2Trend'
2025-07-23 12:51:07 [INFO   ] Encontrado no registry de modelos legados.
2025-07-23 12:51:08 [INFO   ] training_mode: traditional
2025-07-23 12:51:08 [INFO   ] Criando modelo para arquitetura: 'Seq2Trend'
2025-07-23 12:51:08 [INFO   ] Encontrado no registry de modelos legados.
2025-07-23 12:51:15 [INFO   ] Criando modelo para arquitetura: 'Seq2Trend'
2025-07-23 12:51:15 [INFO   ] Encontrado no registry de modelos legados.
2025-07-23 12:51:23 [INFO   ] Criando modelo para arquitetura: 'Seq2Trend'
2025-07-23 12:51:23 [INFO   ] Encontrado no registry de modelos legados.
2025-07-23 12:51:40 [INFO   ] Trend Contribution: 73.21% | Physics Contribution: 26.79%
2025-07-23 12:51:40 [INFO   ] ← Chunk complete, 1 models aggregated
2025-07

KeyboardInterrupt: 

2025-07-23 12:51:52 [INFO   ] training_mode: traditional
2025-07-23 12:51:52 [INFO   ] Criando modelo para arquitetura: 'Seq2Trend'
2025-07-23 12:51:52 [INFO   ] Encontrado no registry de modelos legados.
